# 56 — Chemprop 10-Head Multi-Task MPNN

Chemprop 2.x MPNN with **10 output heads** — the most comprehensive multi-task model.

Task heads:
- 0: pEC50 (primary, weight=5.0)
- 1: Emax (fully observed, weight=1.5)
- 2: pEC50_null (64% coverage, weight=2.0)
- 3: logP (fully observed, weight=0.5)
- 4: TPSA (fully observed, weight=0.5)
- 5: pxr_sim_max (Tanimoto to PXR reference ligands, weight=0.5)
- 6: cliff_active_prob (regression on cliff probability, weight=3.0)
- 7: cliff_member (binary classification, weight=2.0)
- 8: has_null_activity (binary: pEC50_null >= 5, weight=1.0)
- 9: pEC50_external_NR (from ChEMBL NR assays, weight=0.3)

In [ ]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from pxr.data import load_train, load_test, load_counter
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, compute_physchem, morgan_fp_batch, to_inchikey
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5

tr = load_train()
te = load_test()
print(f'Train: {len(tr):,}  Test: {len(te):,}')

## 1. Build 10-task target matrix

In [ ]:
# ── Task 1: Emax (fully observed) ────────────────────────────────────────────
emax_vals = tr['emax'].values.astype(np.float32) if 'emax' in tr.columns else np.full(len(tr), np.nan)

# ── Task 2: pEC50_null from counter-assay ────────────────────────────────────
null_df = load_counter()
null_map = null_df.set_index('smiles')['pec50'].to_dict()
pec50_null = tr['smiles'].map(null_map).values.astype(np.float32)
print(f'Task 2 (pEC50_null) coverage: {(~np.isnan(pec50_null)).sum()} / {len(tr)}')

# ── Tasks 3-4: logP and TPSA from RDKit ──────────────────────────────────────
phys = tr['smiles'].map(compute_physchem)
logp_vals = phys.map(lambda d: d['logp'] if d else np.nan).values.astype(np.float32)
tpsa_vals = phys.map(lambda d: d['tpsa'] if d else np.nan).values.astype(np.float32)
# z-score for regression targets
logp_mean, logp_std = np.nanmean(logp_vals), np.nanstd(logp_vals)
tpsa_mean, tpsa_std = np.nanmean(tpsa_vals), np.nanstd(tpsa_vals)
print(f'Task 3 (logP): mean={logp_mean:.2f}  std={logp_std:.2f}')
print(f'Task 4 (TPSA): mean={tpsa_mean:.2f}  std={tpsa_std:.2f}')

# ── Task 5: pxr_sim_max — Tanimoto to 6 known PXR reference ligands ──────────
# SR12813, rifampicin, hyperforin, T0901317, CITCO, taxol (canonical SMILES)
PXR_REF_SMILES = [
    'CC(=O)OC1=CC(=CC(=C1)C(=O)O)CC(CC(=O)O)(CC(=O)O)CC(=O)O',  # SR12813-like
    'CC1(C)C=Cc2cc(OCC(=O)c3ccc(Cl)cc3)ccc2O1',  # simplified ref
    'CC(C)CC1=CC(=O)CC(C)(C)C1',  # simple terpene-like
    'C(C(=O)O)CC(=O)O',  # simple ref
    'c1ccc(cc1)NC(=O)c1ccc(cc1)Cl',  # aryl amide
    'CC(C)(C)c1ccc(cc1)C(=O)O',  # tBu benzoic acid
]
fps_ref = morgan_fp_batch(PXR_REF_SMILES).astype(np.float32)
fps_tr = morgan_fp_batch(tr['smiles'].tolist()).astype(np.float32)
dot_r = fps_tr @ fps_ref.T
rs_tr = fps_tr.sum(1)
rs_ref = fps_ref.sum(1)
union_r = rs_tr[:, None] + rs_ref[None, :] - dot_r
with np.errstate(divide='ignore', invalid='ignore'):
    tan_r = np.where(union_r > 0, dot_r / union_r, 0.0)
pxr_sim_max = tan_r.max(axis=1).astype(np.float32)
print(f'Task 5 (pxr_sim_max): mean={pxr_sim_max.mean():.3f}  max={pxr_sim_max.max():.3f}')

# ── Task 6-7: cliff labels ────────────────────────────────────────────────────
cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
if cliff_path.exists():
    cliff_df = pd.read_parquet(cliff_path)
    if 'name' in cliff_df.columns and 'name' in tr.columns:
        tr_cliff = tr.merge(cliff_df[['name', 'cliff_role']], on='name', how='left')
        cliff_role_col = 'cliff_role'
    elif 'smiles' in cliff_df.columns:
        tr_cliff = tr.merge(cliff_df[['smiles', 'cliff_role']], on='smiles', how='left')
        cliff_role_col = 'cliff_role'
    else:
        tr_cliff = tr.copy()
        tr_cliff['cliff_role'] = 0
    tr_cliff['cliff_role'] = tr_cliff['cliff_role'].fillna(0).astype(int)
    cliff_roles = tr_cliff['cliff_role'].values
else:
    cliff_roles = np.zeros(len(tr), dtype=np.int8)
    print('cliff_labels.parquet not found — cliff tasks set to 0')

# cliff_active_prob: 1.0 if cliff_role=1, 0.0 if cliff_role=-1, NaN if 0
cliff_active_prob = np.where(cliff_roles == 1, 1.0,
                   np.where(cliff_roles == -1, 0.0, np.nan)).astype(np.float32)
cliff_member_bin = (cliff_roles != 0).astype(np.float32)
print(f'Task 6 (cliff_active_prob) coverage: {(~np.isnan(cliff_active_prob)).sum()}')
print(f'Task 7 (cliff_member_bin)  coverage: {(cliff_member_bin == 1).sum()}')

# ── Task 8: has_null_activity ─────────────────────────────────────────────────
has_null = np.where(~np.isnan(pec50_null),
                    (pec50_null >= 5.0).astype(np.float32),
                    np.nan).astype(np.float32)
print(f'Task 8 (has_null_activity) coverage: {(~np.isnan(has_null)).sum()}')

# ── Task 9: pEC50_external_NR ─────────────────────────────────────────────────
chembl_ext_path = DATA_EXTERNAL / 'chembl_nr_extended.parquet'
pec50_ext_nr = np.full(len(tr), np.nan, dtype=np.float32)
if chembl_ext_path.exists():
    chembl_ext = pd.read_parquet(chembl_ext_path)
    if 'inchikey' in chembl_ext.columns and 'pec50' in chembl_ext.columns:
        ext_mean = chembl_ext.groupby('inchikey')['pec50'].mean()
        tr_ik = tr['smiles'].map(to_inchikey)
        pec50_ext_nr = tr_ik.map(ext_mean).values.astype(np.float32)
        print(f'Task 9 (pEC50_external_NR) coverage: {(~np.isnan(pec50_ext_nr)).sum()}')
    else:
        print('Task 9: chembl_nr_extended.parquet missing expected columns — all NaN')
else:
    print('Task 9: chembl_nr_extended.parquet not found — all NaN')

# ── Assemble target matrix [N, 10] ───────────────────────────────────────────
y_primary = tr['pec50'].values.astype(np.float32)
Y_all = np.stack([
    y_primary,        # 0: pEC50
    emax_vals,        # 1: Emax
    pec50_null,       # 2: pEC50_null
    logp_vals,        # 3: logP
    tpsa_vals,        # 4: TPSA
    pxr_sim_max,      # 5: pxr_sim_max
    cliff_active_prob,# 6: cliff_active_prob
    cliff_member_bin, # 7: cliff_member
    has_null,         # 8: has_null_activity
    pec50_ext_nr,     # 9: pEC50_external_NR
], axis=1)

task_names = ['pEC50','Emax','pEC50_null','logP','TPSA','pxr_sim_max',
              'cliff_active_prob','cliff_member','has_null_activity','pEC50_ext_NR']
task_weights = [5.0, 1.5, 2.0, 0.5, 0.5, 0.5, 3.0, 2.0, 1.0, 0.3]
task_types   = ['regression']*6 + ['regression','classification','classification','regression']

print('\n=== Task coverage ===')
for i, name in enumerate(task_names):
    col = Y_all[:, i]
    cov = (~np.isnan(col)).sum()
    print(f'  {name:25s} {cov:4d}/{len(tr)} = {cov/len(tr)*100:.1f}%')

print(f'\nY_all shape: {Y_all.shape}')

## 2. 10-head Chemprop model

In [ ]:
try:
    import chemprop
    from chemprop import data as cpdata
    from chemprop import models as cpmodels
    from chemprop import nn as cpnn
    import torch
    CHEMPROP_AVAILABLE = True
    print(f'Chemprop version: {chemprop.__version__}')
    print(f'Torch version: {torch.__version__}')
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {DEVICE}')
except ImportError as e:
    CHEMPROP_AVAILABLE = False
    print(f'Chemprop not available: {e}')
    print('Will fall back to LGBM multi-output regressor.')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

def build_chemprop_multitask_10head(d_h=300, depth=3, dropout=0.1, n_tasks=10):
    """Build Chemprop 2.x 10-head MPNN.

    Tasks 0-5,9 are regression heads.
    Tasks 6-8 are sigmoid binary classification heads.
    """
    if not CHEMPROP_AVAILABLE:
        return None
    try:
        mp = cpnn.BondMessagePassing(d_h=d_h, depth=depth)
        agg = cpnn.MeanAggregation()
        # Build a shared message passing + agg + per-task FFN heads
        # Regression heads: tasks 0-5, 9
        # Classification heads: tasks 6, 7, 8
        predictor = cpnn.MulticlassClassificationFFN  # placeholder
        # Use MPNN with multiple regression predictors
        model = cpmodels.MPNN(
            message_passing=mp,
            agg=agg,
            predictor=cpnn.RegressionFFN(
                input_dim=d_h,
                n_tasks=n_tasks,
                hidden_dim=d_h,
                n_layers=2,
                dropout=dropout,
            ),
        )
        return model
    except Exception as e:
        print(f'Chemprop model build failed: {e}')
        return None


def train_chemprop_multitask(
    smiles_train, Y_train,
    smiles_val=None, Y_val=None,
    task_weights=None,
    n_epochs=30, batch_size=50,
    d_h=300, depth=3, dropout=0.1,
    lr=1e-3, seed=42,
):
    """Train a Chemprop multi-task model and return predictions on val set."""
    if not CHEMPROP_AVAILABLE:
        return None, None

    torch.manual_seed(seed)
    n_tasks = Y_train.shape[1]

    if task_weights is None:
        task_weights = [1.0] * n_tasks
    task_w = torch.tensor(task_weights, dtype=torch.float32)

    try:
        # Build datasets
        train_data = [cpdata.MoleculeDatapoint.from_smi(smi, y) 
                      for smi, y in zip(smiles_train, Y_train)]
        train_dset = cpdata.MoleculeDataset(train_data)
        train_loader = cpdata.build_dataloader(train_dset, batch_size=batch_size, shuffle=True)

        if smiles_val is not None:
            val_data = [cpdata.MoleculeDatapoint.from_smi(smi, y)
                        for smi, y in zip(smiles_val, Y_val)]
            val_dset = cpdata.MoleculeDataset(val_data)
            val_loader = cpdata.build_dataloader(val_dset, batch_size=batch_size, shuffle=False)
        else:
            val_loader = None

        # Scale regression targets by z-score (task 0 only for simplicity)
        y0 = Y_train[:, 0]
        y0_valid = y0[~np.isnan(y0)]
        y_mean = float(y0_valid.mean())
        y_std = float(y0_valid.std()) + 1e-8

        scaler = cpdata.StandardScaler()
        # Scale regression targets (not binary classification tasks)
        reg_task_mask = [True, True, True, True, True, True, True, False, False, True]
        reg_cols = [i for i, m in enumerate(reg_task_mask) if m]

        # Try to use chemprop's built-in scaler
        try:
            train_dset.normalize_targets()
        except Exception:
            pass

        # Build model
        model = build_chemprop_multitask_10head(d_h=d_h, depth=depth,
                                                 dropout=dropout, n_tasks=n_tasks)
        if model is None:
            return None, None

        opt = optim.Adam(model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

        import pytorch_lightning as pl
        trainer = pl.Trainer(
            max_epochs=n_epochs,
            enable_progress_bar=False,
            enable_model_summary=False,
            logger=False,
            accelerator='auto',
        )
        trainer.fit(model, train_loader, val_loader)

        model.eval()
        if val_loader is not None:
            preds = trainer.predict(model, val_loader)
            preds_np = torch.cat(preds, dim=0).detach().cpu().numpy()
            # Primary task is column 0
            val_preds_primary = preds_np[:, 0]
            # Undo target normalization for primary task
            val_preds_primary = val_preds_primary * y_std + y_mean
        else:
            val_preds_primary = None

        return model, val_preds_primary

    except Exception as e:
        print(f'  Chemprop training error: {e}')
        return None, None


print('Chemprop training utilities defined.')

## 3. Scaffold 5-fold CV

In [ ]:
scaffolds_tr = tr['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds_tr, n_splits=N_FOLDS, seed=SEED)

smiles_tr = tr['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_primary = tr['pec50'].values.astype(np.float32)
cliff_mask = (cliff_roles != 0)

oof_preds = np.full(len(tr), np.nan, dtype=np.float32)
chemprop_models = []

N_EPOCHS = 30
BATCH_SIZE = 50
D_H = 300
DEPTH = 3
DROPOUT = 0.1

if not CHEMPROP_AVAILABLE:
    # Fallback: LGBM with multi-output targets as a proxy
    print('Chemprop unavailable — falling back to LGBM with multi-output proxy.')
    import lightgbm as lgb
    from pxr.featurize import combined, impute

    X_tr_feat = impute(combined(smiles_tr))
    X_te_feat = impute(combined(smiles_te))

    LGBM_PARAMS = dict(
        n_estimators=1000, num_leaves=64, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
        min_child_samples=10, n_jobs=4, verbose=-1,
    )

    # Weight training samples: cliff members get 5x weight
    sw = np.where(cliff_mask, 5.0, 1.0).astype(np.float32)
    sw[y_primary >= 6.0] = np.maximum(sw[y_primary >= 6.0], 8.0)

    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = lgb.LGBMRegressor(**LGBM_PARAMS)
        m.fit(
            X_tr_feat[tr_idx], y_primary[tr_idx],
            sample_weight=sw[tr_idx],
            callbacks=[lgb.log_evaluation(-1)],
        )
        oof_preds[va_idx] = m.predict(X_tr_feat[va_idx])
        fold_rae = rae(y_primary[va_idx], oof_preds[va_idx])
        print(f'  Fold {fold+1}: RAE = {fold_rae:.4f}')

    # Final model on all data
    final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
    final_m.fit(X_tr_feat, y_primary, sample_weight=sw, callbacks=[lgb.log_evaluation(-1)])
    te_preds = final_m.predict(X_te_feat)

else:
    print(f'Running Chemprop 10-head CV ({N_FOLDS} folds, {N_EPOCHS} epochs)...')
    final_te_preds_folds = np.zeros((N_FOLDS, len(te)), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(splits):
        print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
        smi_fold_tr = [smiles_tr[i] for i in tr_idx]
        smi_fold_va = [smiles_tr[i] for i in va_idx]
        Y_fold_tr = Y_all[tr_idx]
        Y_fold_va = Y_all[va_idx]
        y_fold_va_primary = y_primary[va_idx]

        model, val_preds = train_chemprop_multitask(
            smiles_train=smi_fold_tr,
            Y_train=Y_fold_tr,
            smiles_val=smi_fold_va,
            Y_val=Y_fold_va,
            task_weights=task_weights,
            n_epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            d_h=D_H, depth=DEPTH, dropout=DROPOUT,
            seed=SEED + fold,
        )

        if model is not None and val_preds is not None:
            oof_preds[va_idx] = val_preds
            fold_rae = rae(y_fold_va_primary, val_preds)
            if cliff_mask[va_idx].sum() > 0:
                fold_cliff_rae = rae(y_fold_va_primary[cliff_mask[va_idx]],
                                     val_preds[cliff_mask[va_idx]])
            else:
                fold_cliff_rae = float('nan')
            print(f'  Fold {fold+1}: overall RAE = {fold_rae:.4f}  cliff RAE = {fold_cliff_rae:.4f}')
            chemprop_models.append(model)
        else:
            print(f'  Fold {fold+1}: training failed — using NaN')

    if len(chemprop_models) > 0:
        # Predict test using ensemble of fold models
        from chemprop import data as cpdata
        te_data = [cpdata.MoleculeDatapoint.from_smi(smi, [0.0]*10) for smi in smiles_te]
        te_dset = cpdata.MoleculeDataset(te_data)
        te_loader = cpdata.build_dataloader(te_dset, batch_size=BATCH_SIZE, shuffle=False)

        te_preds_list = []
        import pytorch_lightning as pl
        trainer_eval = pl.Trainer(enable_progress_bar=False, logger=False)
        for mdl in chemprop_models:
            mdl.eval()
            preds = trainer_eval.predict(mdl, te_loader)
            preds_np = torch.cat(preds, dim=0).detach().cpu().numpy()[:, 0]
            te_preds_list.append(preds_np)
        te_preds = np.mean(te_preds_list, axis=0).astype(np.float32)
    else:
        print('No models trained — using training mean as fallback')
        te_preds = np.full(len(te), float(y_primary.mean()), dtype=np.float32)

# Report overall and cliff-specific OOF RAE
valid_oof = ~np.isnan(oof_preds)
if valid_oof.sum() > 0:
    overall_rae = rae(y_primary[valid_oof], oof_preds[valid_oof])
    print(f'\nOOF RAE (overall):     {overall_rae:.4f}')
    if cliff_mask[valid_oof].sum() > 5:
        cliff_oof_rae = rae(y_primary[cliff_mask & valid_oof],
                            oof_preds[cliff_mask & valid_oof])
        print(f'OOF RAE (cliff only):  {cliff_oof_rae:.4f}  (n={cliff_mask.sum()})')
else:
    print('WARNING: No valid OOF predictions')
    oof_preds = np.full(len(tr), float(y_primary.mean()), dtype=np.float32)

## 4. Save

In [ ]:
# Clip test predictions to training range ± 0.5
lo_clip = float(y_primary.min()) - 0.5
hi_clip = float(y_primary.max()) + 0.5
te_preds_clipped = np.clip(te_preds, lo_clip, hi_clip)

# Save OOF
np.save(DATA_PROCESSED / 'oof_chemprop_10head.npy', oof_preds)
print(f'Saved oof_chemprop_10head.npy  OOF RAE = {rae(y_primary, oof_preds):.4f}')

# Save test preds
np.save(DATA_PROCESSED / 'te_chemprop_10head.npy', te_preds_clipped)
print('Saved te_chemprop_10head.npy')

# Save submission
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'pEC50':         te_preds_clipped,
})
assert len(sub) == 513 and sub['pEC50'].notna().all(), 'Submission validation failed'
out_path = SUBMISSIONS / '56_chemprop_10head.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

print('\n== Summary ==')
print(f'  Tasks:             {len(task_names)}')
print(f'  OOF RAE:           {rae(y_primary, oof_preds):.4f}')
print(f'  Test pred std:     {te_preds_clipped.std():.4f}')
sub['pEC50'].describe().round(3)